# UC01 — Puntuación de Fraude en Siniestros

## Objetivos de Aprendizaje

1. **Crear un flujo ML completo** usando solo SQL en Snowflake
2. **Usar `SNOWFLAKE.ML.CLASSIFICATION`** para construir un modelo de clasificación binaria
3. **Generar datos sintéticos** que simulan siniestros reales de seguros
4. **Construir variables predictoras** a partir de múltiples fuentes de datos
5. **Evaluar el rendimiento del modelo** con métricas integradas
6. **Puntuar nuevos siniestros** y generar predicciones en tiempo real
7. **Construir dashboards interactivos** con Streamlit

---

## Qué Construirás

Un **modelo de clasificación binaria** que predice si un siniestro es fraudulento.

**Valor de negocio**: Reducir pérdidas por fraude hasta un 40%.

---

## Paso 1: Configuración del Entorno

In [ ]:
CREATE DATABASE IF NOT EXISTS FRAUDE_SINIESTROS_DB;
CREATE SCHEMA IF NOT EXISTS FRAUDE_SINIESTROS_DB.PUBLIC;
USE SCHEMA FRAUDE_SINIESTROS_DB.PUBLIC;

CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH
    WITH WAREHOUSE_SIZE = 'SMALL' AUTO_SUSPEND = 60 AUTO_RESUME = TRUE;
USE WAREHOUSE COMPUTE_WH;

---

## Paso 2: Generar Datos Sintéticos de Tomadores

In [ ]:
CREATE OR REPLACE TABLE tomadores AS
SELECT
    'TOM' || LPAD(SEQ4(), 5, '0') AS tomador_id,
    UNIFORM(22, 75, RANDOM()) AS edad,
    CASE WHEN UNIFORM(0,1,RANDOM()) < 0.5 THEN 'M' ELSE 'F' END AS genero,
    UNIFORM(1, 30, RANDOM()) AS antiguedad_anos,
    CASE MOD(UNIFORM(1,100,RANDOM()),4)
        WHEN 0 THEN 'Auto' WHEN 1 THEN 'Hogar' WHEN 2 THEN 'Vida' ELSE 'Salud'
    END AS tipo_poliza,
    UNIFORM(0, 5, RANDOM()) AS siniestros_previos
FROM TABLE(GENERATOR(ROWCOUNT => 500));

SELECT * FROM tomadores LIMIT 10;

---

## Paso 3: Generar Siniestros con Indicadores de Fraude

In [ ]:
CREATE OR REPLACE TABLE siniestros AS
SELECT
    'SIN' || LPAD(SEQ4(), 6, '0') AS siniestro_id,
    t.tomador_id,
    CASE WHEN UNIFORM(0,1,RANDOM()) < 0.20 THEN 1 ELSE 0 END AS es_fraude,
    DATEADD(day, -UNIFORM(1, 365, RANDOM()), CURRENT_DATE()) AS fecha_siniestro,
    CASE WHEN es_fraude = 1 THEN UNIFORM(15, 60, RANDOM()) ELSE UNIFORM(1, 10, RANDOM()) END AS dias_hasta_notificacion,
    CASE WHEN es_fraude = 1 THEN UNIFORM(8000, 50000, RANDOM()) ELSE UNIFORM(200, 12000, RANDOM()) END::DECIMAL(10,2) AS importe_declarado,
    CASE WHEN es_fraude = 1 THEN UNIFORM(0, 1, RANDOM()) ELSE UNIFORM(1, 4, RANDOM()) END AS num_testigos,
    UNIFORM(0, 23, RANDOM()) AS hora_accidente,
    CASE WHEN es_fraude = 1 AND UNIFORM(0,1,RANDOM()) < 0.7 THEN 1 ELSE 0 END AS taller_sospechoso,
    CASE WHEN es_fraude = 1 THEN UNIFORM(3, 8, RANDOM()) ELSE UNIFORM(0, 2, RANDOM()) END::FLOAT AS ratio_siniestros_ano
FROM tomadores t CROSS JOIN TABLE(GENERATOR(ROWCOUNT => 2))
LIMIT 1000;

SELECT es_fraude, COUNT(*) AS total, ROUND(AVG(importe_declarado),2) AS media_importe
FROM siniestros GROUP BY es_fraude;

---

## Paso 4: Ingeniería de Variables

In [ ]:
CREATE OR REPLACE TABLE features_fraude AS
SELECT
    s.siniestro_id,
    t.edad::FLOAT AS edad,
    t.antiguedad_anos::FLOAT AS antiguedad,
    t.siniestros_previos::FLOAT AS siniestros_previos,
    s.dias_hasta_notificacion::FLOAT AS dias_notificacion,
    s.importe_declarado::FLOAT AS importe,
    s.num_testigos::FLOAT AS testigos,
    s.hora_accidente::FLOAT AS hora,
    s.taller_sospechoso::FLOAT AS taller_sospechoso,
    s.ratio_siniestros_ano AS ratio_siniestros,
    s.es_fraude
FROM siniestros s JOIN tomadores t ON s.tomador_id = t.tomador_id;

SELECT * FROM features_fraude LIMIT 10;

---

## Paso 5: Dividir en Train/Test

In [ ]:
CREATE OR REPLACE TABLE entrenamiento AS SELECT * FROM features_fraude SAMPLE (80);
CREATE OR REPLACE TABLE test AS SELECT * FROM features_fraude WHERE siniestro_id NOT IN (SELECT siniestro_id FROM entrenamiento);

SELECT 'Entrenamiento' AS conjunto, COUNT(*) AS total, SUM(es_fraude) AS fraudes, ROUND(SUM(es_fraude)/COUNT(*)*100,1)||'%' AS tasa FROM entrenamiento
UNION ALL
SELECT 'Test', COUNT(*), SUM(es_fraude), ROUND(SUM(es_fraude)/COUNT(*)*100,1)||'%' FROM test;

---

## Paso 6: Entrenar Modelo de Detección de Fraude

In [ ]:
CREATE OR REPLACE SNOWFLAKE.ML.CLASSIFICATION detector_fraude(
    INPUT_DATA     => SYSTEM$REFERENCE('TABLE', 'entrenamiento'),
    TARGET_COLNAME => 'es_fraude',
    CONFIG_OBJECT  => {'evaluate': TRUE}
);

---

## Paso 7: Evaluar Rendimiento

In [ ]:
CALL detector_fraude!SHOW_EVALUATION_METRICS();

In [ ]:
CALL detector_fraude!SHOW_CONFUSION_MATRIX();

In [ ]:
CALL detector_fraude!SHOW_FEATURE_IMPORTANCE();

---

## Paso 8: Puntuar Siniestros

- **Alto Riesgo** (≥70%): Investigación inmediata
- **Riesgo Medio** (30-70%): Revisión manual
- **Bajo Riesgo** (<30%): Aprobación automática

In [ ]:
CREATE OR REPLACE TABLE predicciones_fraude AS
SELECT
    siniestro_id,
    detector_fraude!PREDICT(OBJECT_CONSTRUCT(
        'edad', edad, 'antiguedad', antiguedad, 'siniestros_previos', siniestros_previos,
        'dias_notificacion', dias_notificacion, 'importe', importe, 'testigos', testigos,
        'hora', hora, 'taller_sospechoso', taller_sospechoso, 'ratio_siniestros', ratio_siniestros
    )) AS prediccion,
    prediccion['class']::INT AS fraude_predicho,
    prediccion['probability']['1']::FLOAT AS prob_fraude,
    CASE
        WHEN prediccion['probability']['1']::FLOAT >= 0.70 THEN 'Alto Riesgo'
        WHEN prediccion['probability']['1']::FLOAT >= 0.30 THEN 'Riesgo Medio'
        ELSE 'Bajo Riesgo'
    END AS nivel,
    es_fraude AS fraude_real
FROM test;

SELECT siniestro_id, ROUND(prob_fraude,3) AS prob, nivel, fraude_real
FROM predicciones_fraude ORDER BY prob_fraude DESC LIMIT 20;

---

## Paso 9: Dashboard Interactivo

In [ ]:
import streamlit as st
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()
st.title('Detección de Fraude en Siniestros')
st.markdown('### Puntuación ML en Tiempo Real — Snowflake Cortex')

df = session.table('predicciones_fraude').to_pandas()

c1, c2, c3, c4 = st.columns(4)
with c1: st.metric('Total Siniestros', len(df))
with c2: st.metric('Alto Riesgo', len(df[df['NIVEL']=='Alto Riesgo']))
with c3: st.metric('Exactitud', f"{(df['FRAUDE_PREDICHO']==df['FRAUDE_REAL']).mean():.1%}")
with c4: st.metric('Tasa Fraude Real', f"{df['FRAUDE_REAL'].mean():.1%}")

st.markdown('---')
st.subheader('Distribución por Nivel de Riesgo')
rc = df['NIVEL'].value_counts()
st.bar_chart(pd.DataFrame({'Siniestros': rc.values}, index=rc.index))

st.markdown('---')
st.subheader('Siniestros Sospechosos')
filtro = st.multiselect('Filtrar por nivel', ['Alto Riesgo','Riesgo Medio','Bajo Riesgo'], default=['Alto Riesgo','Riesgo Medio'])
fdf = df[df['NIVEL'].isin(filtro)].sort_values('PROB_FRAUDE', ascending=False)
fdf['Prob %'] = fdf['PROB_FRAUDE'].apply(lambda x: f'{x:.1%}')
st.dataframe(fdf[['SINIESTRO_ID','Prob %','NIVEL','FRAUDE_PREDICHO','FRAUDE_REAL']], use_container_width=True, height=400)

st.caption('Desarrollado con Snowflake Cortex ML + Streamlit | Datos: Sintéticos')

---

## Paso 10: Limpieza (Opcional)

In [ ]:
-- DROP DATABASE IF EXISTS FRAUDE_SINIESTROS_DB;
-- DROP WAREHOUSE IF EXISTS COMPUTE_WH;